<a href="https://colab.research.google.com/github/snehavagu-source/datamining-proj/blob/main/dm_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Datasets ni load cheyadam
df = pd.read_csv('dm1 (1).csv') # Review data
data1 = pd.read_csv('dm2.csv') # Customer data

# 2. Duplicates ni handle cheyadam (Data Integration rule)
data1 = data1.drop_duplicates(subset='reviewID')

# 3. Dropna - Deeni valla empty rows unte pothayi
# Kaani be careful: data ekkuva loss avthundante subset=['reviewID'] pettandi
data1 = data1.dropna(subset=['reviewID'])

# 4. Merging (Joining on reviewID)
# 'inner' join vaadithe renditlo common ga unna reviewIDs matrame vastayi
merged_data = pd.merge(df, data1, on='reviewID', how='inner')

# 5. Result ni save cheyadam
merged_data.to_csv("merged_dataset.csv", index=False)

# 6. Result ni display cheyadam
print("Merged Dataset Shape:", merged_data.shape)

In [ ]:
# Step 2: Data Cleaning (Unit 2)
# Numeric columns lo unna empty values ni median tho fill chestunnam
# Ide step manaki WEKA lo error raakunda help chestundi
final_weka_data = merged_data.fillna(merged_data.median(numeric_only=True))

# Inka emaina string columns lo missing values unte 'Unknown' tho fill cheyandi
final_weka_data = final_weka_data.fillna('Unknown')

# File save cheyadam
final_weka_data.to_csv('final_weka_data.csv', index=False)
print("Cleaned 'final_weka_data.csv' is ready for WEKA!")

In [ ]:
import pandas as pd

# 1. Dataset load cheyadam (Meeru pampina final file)
df = pd.read_csv('final_weka_data.csv')

# 2. SAFE STEP: Column names lo spaces unte remove chestunnam
# Deeni valla 'KeyError' poyi perfect ga pani chestundi
df.columns = df.columns.str.strip()

# 3. Sentiment_Label create chese logic
def get_label(score):
    try:
        score = float(score) # Score numeric ga lekapothe convert chestunnam
        if score > 0.05: return 'Positive'
        elif score < -0.05: return 'Negative'
        else: return 'Neutral'
    except:
        return 'Neutral' # Error vaste Neutral ga isthunnam

# Use 'sentiment_score_x' which is available in the merged_data
sentiment_col_name = 'sentiment_score_x'

if sentiment_col_name in df.columns:
    df['Sentiment_Label'] = df[sentiment_col_name].apply(get_label)
    print("Sentiment_Label added successfully!")
    # 4. WEKA ki ready ga CSV save cheyadam
    # quoting=1 pedithe WEKA confusion lekunda load avthundi
    df.to_csv("Ready_For_Weka (1).csv", index=False, quoting=1)

    # Preview results
    print("\nFinal Labels Count:\n", df['Sentiment_Label'].value_counts())
    display(df[[sentiment_col_name, 'Sentiment_Label']].head())
else:
    print(f"Error: '{sentiment_col_name}' column dorakaledu. Please check file columns.")
    print("Available columns:", df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Dataset upload cheyandi (final_weka_data.csv)
df = pd.read_csv('Ready_For_Weka (1).csv')
print("Data Loaded Successfully!")
df.head(20)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('final_weka_data.csv') # Changed filename to load the correct input
df.columns = df.columns.str.strip() # Clean column names for consistency

# ---------------- DATA TRANSFORMATION ----------------

# 1. Handling Missing Values
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Membership_Years_y'] = df['Membership_Years_y'].fillna(df['Membership_Years_y'].mean()) # Impute any missing 'Membership_Years_y'

# 2. Remove Duplicates
df = df.drop_duplicates()

# 3. Type Conversion (Example - these columns might not exist or need different handling)
# df['Date'] = pd.to_datetime(df['Date'])
# df['Salary'] = df['Salary'].astype(float)

# 4. Feature Creation (Example - assuming 'Bonus' and 'Salary' columns exist)
# df['Total_Salary'] = df['Salary'] + df['Bonus']

# 5. Column Renaming (Example)
# df = df.rename(columns={'Name': 'Employee_Name'})

# 6. Encoding Categorical Data (Example - assuming 'City' column exists)
# If you have non-numeric columns like 'Country', 'City', 'Gender', etc., encode them
for col in df.select_dtypes(include=['object']).columns:
    if col not in ['productASIN', 'productVariant', 'reviewMetadata', 'reviewTitle', 'reviewURL', 'Country_x', 'City_x', 'Country_y', 'City_y', 'Gender', 'Sentiment_Label']:# Exclude identifier columns and the target if already present
        df = pd.get_dummies(df, columns=[col], prefix=col, drop_first=True)

# 7. Normalization (Min-Max Scaling - Example for 'Salary')
# if 'Salary' in df.columns:
#     df['Salary_Scaled'] = (df['Salary'] - df['Salary'].min()) / (df['Salary'].max() - df['Salary'].min())

# 8. Standardization (Example for 'Salary')
# if 'Salary' in df.columns:
#     df['Salary_Std'] = (df['Salary'] - df['Salary'].mean()) / df['Salary'].std()

# 9. Filtering Data (Example)
# df = df[df['Salary'] > 30000]

# 10. Sorting (Example)
# df = df.sort_values(by='Salary', ascending=False)

df.to_csv("Transformed_Data.csv", index=False) # Save the transformed data

# Final Output
print("Data Transformation Complete. Shape:", df.shape)
df.head()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2, f_classif # Import f_classif
from sklearn.feature_selection import VarianceThreshold

# Load data
df = pd.read_csv("Ready_For_Weka (1).csv") # Changed to load the correct file
df.columns = df.columns.str.strip() # Clean column names

# ---------------- DATA REDUCTION ----------------

# 1. Remove duplicate rows
df = df.drop_duplicates()

# 2. Handle missing values
df = df.dropna()

# 3. Feature Selection (SelectKBest)
y = df['Sentiment_Label']  # Using 'Sentiment_Label' as the target column
X = df.select_dtypes(include=[np.number]).drop(columns=['Sentiment_Label'], errors='ignore')

# Convert y to numerical labels if needed for certain score_funcs (f_classif can handle categorical directly)
y_encoded, y_labels = pd.factorize(y)

# Changed score_func from chi2 to f_classif
selector = SelectKBest(score_func=f_classif, k=5)
X_new = selector.fit_transform(X, y_encoded)

# Convert back to DataFrame
selected_columns = X.columns[selector.get_support()]
X_new = pd.DataFrame(X_new, columns=selected_columns)

# 4. Remove low variance features
var_thresh = VarianceThreshold(threshold=0.01) # Use a more relevant threshold for scaled data
X_var = var_thresh.fit_transform(X_new)

selected_var_cols = selected_columns[var_thresh.get_support()]
X_var = pd.DataFrame(X_var, columns=selected_var_cols)

# 5. Dimensionality Reduction using PCA
pca = PCA(n_components=3) # Changed n_components to 3 for better representation
X_pca = pca.fit_transform(X_var)

# Convert PCA result to DataFrame
df_reduced = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])

# Final Output
print("Data Reduction Complete. Shape:", df_reduced.shape)
df_reduced.head()

In [ ]:
#Churn prediction(J48 Algorithm)

import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Load data
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

# Target
y = df['Sentiment_Label']

# Features
X = df.select_dtypes(include=[np.number])
X = X.drop(columns=['reviewID'], errors='ignore')

# Model (J48 equivalent)
model = DecisionTreeClassifier()

# 🔥 SAME AS WEKA → 10-FOLD CROSS VALIDATION
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

y_pred = cross_val_predict(model, X, y, cv=cv)

# Accuracy
print("Accuracy:", accuracy_score(y, y_pred))

# Confusion Matrix
cm = confusion_matrix(y, y_pred)
labels = model.fit(X, y).classes_

print("\n=== Confusion Matrix ===\n")

print("   ", end="")
for l in labels:
    print(l[:1], end=" ")
print("<-- classified as")

for i, row in enumerate(cm):
    for val in row:
        print(f"{val:4}", end="")
    print(f" | {labels[i]}")

# Detailed report
print("\n\n=== Classification Report ===\n")
print(classification_report(y, y_pred))

# 🌳 Tree Visualization
plt.figure(figsize=(15,10))
plot_tree(model, filled=True, feature_names=X.columns, class_names=model.classes_)
plt.title("Decision Tree - Churn Prediction")
plt.show()

In [ ]:
# ===============================
# SENTIMENT ANALYSIS - NAIVE BAYES
# ===============================

import pandas as pd
import numpy as np
import time

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

# Target
y = df['Sentiment_Label']

# Features (numeric only)
X = df.select_dtypes(include=[np.number])
X = X.drop(columns=['reviewID'], errors='ignore')

# --- FIX: Impute remaining NaNs in X before passing to GaussianNB ---
if X.isnull().sum().sum() > 0:
    print(f"Found {X.isnull().sum().sum()} NaNs in feature data. Imputing with median...")
    X = X.fillna(X.median())

# ===============================
# MODEL (Naive Bayes)
# ===============================
model = GaussianNB()

# ===============================
# 10-FOLD CROSS VALIDATION
# ===============================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

start = time.time()
y_pred = cross_val_predict(model, X, y, cv=cv)
end = time.time()

# Fit full model (for consistency)
model.fit(X, y)

# ===============================
# SUMMARY
# ===============================
total = len(y)
correct = np.sum(y == y_pred)
incorrect = total - correct
accuracy = accuracy_score(y, y_pred) * 100

print("=== Stratified cross-validation ===")
print("=== Summary ===\n")

print(f"Correctly Classified Instances   {correct}     {accuracy:.4f} %")
print(f"Incorrectly Classified Instances {incorrect}     {100-accuracy:.4f} %")
print(f"Total Number of Instances        {total}")

print(f"Time taken to build model: {end-start:.2f} seconds")

# ===============================
# DETAILED ACCURACY
# ===============================
print("\n=== Detailed Accuracy By Class ===\n")
print(classification_report(y, y_pred))

# ===============================
# CONFUSION MATRIX (WEKA STYLE)
# ===============================
cm = confusion_matrix(y, y_pred)
labels = model.classes_

print("\n=== Confusion Matrix ===\n")

print("   ", end="")
for i in range(len(labels)):
    print(chr(97+i), end=" ")
print("<-- classified as")

for i, row in enumerate(cm):
    for val in row:
        print(f"{val:4}", end="")
    print(f" | {chr(97+i)} = {labels[i]}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')

# Clean column names
df.columns = df.columns.str.strip()

# ===============================
# SELECT NUMERIC COLUMNS
# ===============================
numeric_df = df.select_dtypes(include=['number'])

# ===============================
# CORRELATION MATRIX
# ===============================
corr_matrix = numeric_df.corr()

print("Correlation Matrix:\n")
print(corr_matrix)

# ===============================
# HEATMAP VISUALIZATION
# ===============================
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
#Apriori Growth Algorithm

# ===============================
# INSTALL LIBRARY
# ===============================
!pip install mlxtend

# ===============================
# IMPORT LIBRARIES
# ===============================
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

print("Data Loaded Successfully!")
print(df.head())

# ===============================
# PREPROCESSING
# ===============================
# Convert categorical columns to string
df['Sentiment_Label'] = df['Sentiment_Label'].astype(str)

# Select columns (you can add more)
data = df[['Sentiment_Label']]

# One-hot encoding (IMPORTANT)
encoded_data = pd.get_dummies(data)

print("\nEncoded Data:")
print(encoded_data.head())

# ===============================
# APPLY APRIORI
# ===============================
frequent_itemsets = apriori(encoded_data, min_support=0.1, use_colnames=True)

print("\nFrequent Itemsets:\n")
print(frequent_itemsets)

# ===============================
# ASSOCIATION RULES
# ===============================
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print("\nAssociation Rules:\n")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# ===============================
# SAVE OUTPUT
# ===============================
rules.to_csv("apriori_rules_output.csv", index=False)

print("\nRules saved successfully!")

In [ ]:
#FP Growth Algorithm

# ===============================
# INSTALL LIBRARY
# ===============================
!pip install mlxtend

# ===============================
# IMPORT LIBRARIES
# ===============================
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

print("Data Loaded Successfully!")
print(df.head())

# ===============================
# PREPROCESSING
# ===============================
# Convert data into categorical (for association rules)
# Example: convert sentiment into dummy variables

df['Sentiment_Label'] = df['Sentiment_Label'].astype(str)

# Select useful columns (modify based on your dataset)
data = df[['Sentiment_Label']]

# Convert to one-hot encoding
encoded_data = pd.get_dummies(data)

print("\nEncoded Data:")
print(encoded_data.head())

# ===============================
# APPLY FP-GROWTH
# ===============================
frequent_itemsets = fpgrowth(encoded_data, min_support=0.1, use_colnames=True)

print("\nFrequent Itemsets:\n")
print(frequent_itemsets)

# ===============================
# ASSOCIATION RULES
# ===============================
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print("\nAssociation Rules:\n")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# ===============================
# SAVE OUTPUT
# ===============================
rules.to_csv("fpgrowth_rules_output.csv", index=False)

print("\nRules saved successfully!")

In [ ]:
#cluster analysis(kmeans)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

# ===============================
# SELECT NUMERIC FEATURES
# ===============================
X = df.select_dtypes(include=[np.number])

# Remove ID column if exists
X = X.drop(columns=['reviewID'], errors='ignore')

# ===============================
# IMPUTE REMAINING NaNs
# ===============================
# Fill any remaining NaNs in X with the median of their respective columns
# This is crucial as KMeans does not accept missing values
if X.isnull().sum().sum() > 0:
    print(f"Found {X.isnull().sum().sum()} NaNs in feature data. Imputing with median before scaling...")
    X = X.fillna(X.median())

# ===============================
# FEATURE SCALING (IMPORTANT)
# ===============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# APPLY K-MEANS
# ===============================
kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)   # 3 clusters, added n_init to suppress warning
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataset
df['Cluster'] = clusters

print("Cluster Counts:\n")
print(df['Cluster'].value_counts())

In [ ]:
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=clusters)
plt.title("K-Means Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
# ===============================
# HIERARCHICAL CLUSTERING
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv('Ready_For_Weka (1).csv')
df.columns = df.columns.str.strip()

print("Data Loaded Successfully!")
print(df.head())

# ===============================
# SELECT NUMERIC FEATURES
# ===============================
X = df.select_dtypes(include=[np.number])

# Remove ID if exists
X = X.drop(columns=['reviewID'], errors='ignore')

# ===============================
# IMPUTE REMAINING NaNs
# ===============================
# Fill any remaining NaNs in X with the median of their respective columns
# This is crucial as hierarchical clustering does not accept missing values
if X.isnull().sum().sum() > 0:
    print(f"Found {X.isnull().sum().sum()} NaNs in feature data. Imputing with median before scaling...")
    X = X.fillna(X.median())

# ===============================
# DATA SCALING (IMPORTANT)
# ===============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# APPLY HIERARCHICAL CLUSTERING
# ===============================
Z = linkage(X_scaled, method='ward')

# ===============================
# DENDROGRAM
# ===============================
plt.figure(figsize=(10, 6))
dendrogram(Z)
plt.title("Dendrogram")
plt.xlabel("Data Points")
plt.ylabel("Distance")
plt.show()

# ===============================
# FORM CLUSTERS (choose k)
# ===============================
k = 3   # clusters count (change if needed)

clusters = fcluster(Z, k, criterion='maxclust')

# Add cluster labels to dataset
df['Cluster'] = clusters

print("\nCluster Counts:\n")
print(df['Cluster'].value_counts())

# ===============================
# SAVE RESULT
# ===============================
df.to_csv("hierarchical_clusters_output.csv", index=False)

print("\nClustered dataset saved!")